# מעבדה 13 — הפוטנציאל הכימי

במעבדה הזו חלקיקים קופצים באקראי בין קופסאות, ואתם מגלים מה הם חולקים בסוף. זו אינה הצפיפות.
תמדדו את הפוטנציאל הכימי בשלוש דרכים, תראו אותו משתווה בזמן שהצפיפויות נפרדות, תערמו קופסאות
לאטמוספרה, תהפכו קופסה אחת למשטח ספיחה, ותסיימו באומדן של קבוע הגזים מתוך קריאות (סינתטיות)
של לחץ אוסמוטי.

המסלול: המודל והאובייקטים שלו (חלק 1); פוטנציאל כימי אחד, שלוש דרכים (חלק 2); החלפת חלקיקים
בין שתי קופסאות (חלק 3); שני ניסויי הפרכה (חלק 4); הצורה הפונקציונלית של הפיצול (חלק 5); עד כמה
יציב השוויון של הפוטנציאלים הכימיים (חלק 6); העמוד הברומטרי (חלק 7); אתר הספיחה (חלק 8); אוסמוזה
ומדידה (חלק 9); הבדיקות האוטומטיות (חלק 10); וחקירה חופשית (חלק 11).

עברו עליה בסדר. במקום שבו המחברת מבקשת מכם לנבא, רשמו את ניבויכם בתא המיועד לכך **לפני**
הרצת התא הבא. ניבוי שהתחייבתם אליו הוא הדרך האמינה היחידה לגלות שטעיתם.

## מפרט המודל

| | |
|---|---|
| **מערכת** | $N$ חלקיקים זהים, ללא אינטראקציה, בקופסאות של גז סריג; לקופסה $k$ יש $M_k$ אתרים, כל אחד לחלקיק יחיד, באנרגיה $u_k$ |
| **דינמיקה** | חלקיק אקראי מציע קפיצה לאתר אקראי מבין כל האתרים; קפיצה לאתר ריק בקופסה אחרת מתקבלת בהסתברות $\min(1, e^{-\Delta u/\kB T})$ |
| **גבול** | סגורה לחלקיקים כמכלול ($N$ נשמר בדיוק); אמבט סמוי בטמפרטורה $T$ מספק את האנרגיה של כל קפיצה |
| **צבר** | קנוני עבור המערכת כולה; כל קופסה היא בקירוב גרנד-קנונית, והאחרות משמשות לה מאגר חלקיקים |
| **מוזנח** | אינטראקציות מעבר לאכלוס יחיד של אתר, קצבי הקפיצה (הזמן נספר בהצעות), מבנה בתוך קופסה |
| **תקף כאשר** | $M_k, N_k \gg 1$; נוסחאות הגבול המדולל דורשות $N_k \ll M_k$; נוסחאות הגז האידיאלי דורשות $n \ll n_Q$ |
| **אופני כישלון** | קופסאות כמעט מלאות, חלקיקים באינטראקציה (מודול 15), $n \to n_Q$ (מודול 17) |

In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# thermolab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy, matplotlib and sympy, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Add any new *pure-Python* dependency of
# thermolab to the list below.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("thermolab", deps=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from thermolab import chemical, fundamental, potentials
from thermolab.constants import AMU, K_B, N_A, R_GAS
from thermolab.validation import (
    convergence_study,
    relative_error,
    scaling_exponent,
    seed_study,
)

# Every stochastic call below takes an explicit numpy Generator; nothing is seeded globally.

T = 300.0  # K
KT = K_B * T
print(f"k_B T at {T:.0f} K = {KT:.4e} J = {KT / 1.602176634e-19:.4f} eV")

## חלק 1 — האובייקטים

במודול `thermolab.chemical` יש שלושה סוגים של אובייקטים.

| פונקציה | מה היא נותנת |
|---|---|
| `mu_from_entropy(relation, U, V, N)` | הפרש מרכזי על כל $S(U, V, N)$: $-T(\partial S/\partial N)_{U,V}$ |
| `mu_from_gibbs(relation, T, P, N)` | הפרש מרכזי על $G$ של מודול 10: $(\partial G/\partial N)_{T,P}$ |
| `ideal_gas_mu(n, T, m, u_ext)` | $\kB T\ln(n/n_Q) + u_{\mathrm{ext}}$ |
| `two_boxes`, `column` | אוספים של קופסאות גז סריג |
| `particle_exchange_sim(boxes, counts, T, steps, rng)` | סימולציית הקפיצות; רושמת את מספר החלקיקים בכל קופסה |
| `exact_count_distribution(boxes, N, T)` | כל פיצול אפשרי בין שתי קופסאות, במשקלו המדויק |
| `site_occupation(mu, T, epsilon)` | $1/(e^{(\varepsilon - \mu)/\kB T} + 1)$ |

לקופסת גז סריג בת $M$ אתרים, המחזיקה $N$ חלקיקים באנרגיית אתר $u$, יש פוטנציאל כימי
$\mu = u + \kB T\ln[N/(M - N)]$, ובקופסה מדוללת הוא $u + \kB T\ln(N/M)$: אנרגיה, ועוד איבר
אנטרופי שגדל עם הצפיפות.

### לנבא

התחייבו לתשובה לכל אחד מארבעת הניבויים של דף המודול לפני שתמשיכו:

1. שתי קופסאות, הרצפה של B מוגבהת, מתחילות בצפיפויות *שוות*. האם חלקיקים זורמים? לאיזה כיוון?
2. הגביהו את הרצפה של B מהערך $2$ אל הערך $4\,\kB T$. פי כמה משתנה יחס הצפיפויות הסופי?
3. אין רצפות; קופסה אחת גדולה פי עשרה מהשנייה; בהתחלה *מספרים* שווים של חלקיקים. האם חלקיקים
   זורמים?
4. שקית של תמיסת סוכר בתוך מים טהורים, ודופן השקית מעבירה רק מים. לאיזה כיוון נעים המים?

**הניבויים שלכם:**

1.
2.
3.
4.

## חלק 2 — פוטנציאל כימי אחד, שלוש דרכים

ארגון בטמפרטורה $300\ \mathrm{K}$ ובלחץ $1\ \mathrm{bar}$. חשבו את הפוטנציאל הכימי שלו:

- כשיפוע של $S(U, V, N)$ של סאקור–טטרודה, $-T(\partial S/\partial N)_{U,V}$ — כאשר גם $T$ נקראת
  מאותו משטח, כך שלא נעשה שימוש בשום דבר מלבד $S$;
- כשיפוע $(\partial G/\partial N)_{T,P}$ של $G(T, P, N)$ של מודול 10, הנבנית נומרית מאותה $S$;
- מתוך הצורה הסגורה $\kB T\ln(n/n_Q)$.

### לנבא

האם שלוש התוצאות יסכימו? האם $\mu$ יהיה חיובי או שלילי, ובערך כמה יחידות של $\kB T$?

**הניבוי שלכם:**

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
ARGON = 39.95 * AMU
PRESSURE = 1.0e5
N_GAS = 1.0e22
volume = N_GAS * KT / PRESSURE
energy = 1.5 * N_GAS * KT
argon = fundamental.monatomic_ideal_gas(ARGON)

mu_entropy = chemical.mu_from_entropy(argon, energy, volume, N_GAS)
mu_gibbs = chemical.mu_from_gibbs(argon, T, PRESSURE, N_GAS)
mu_closed = chemical.ideal_gas_mu(N_GAS / volume, T, ARGON)
g_per_particle = potentials.gibbs_from(argon, T, PRESSURE, N_GAS) / N_GAS
n_q = chemical.quantum_concentration(T, ARGON)

print(f"n = {N_GAS / volume:.3e} m^-3,  n_Q = {n_q:.3e} m^-3,  n/n_Q = {N_GAS / volume / n_q:.2e}")
print(f"-T dS/dN (entropy slope): {mu_entropy / KT:+.9f} k_B T")
print(f" dG/dN   (Gibbs slope):   {mu_gibbs / KT:+.9f} k_B T")
print(f" k_B T ln(n/n_Q):         {mu_closed / KT:+.9f} k_B T")
print(f" G / N:                   {g_per_particle / KT:+.9f} k_B T")
print(f"relative gaps: {relative_error(mu_entropy, mu_closed):.1e}, "
      f"{relative_error(mu_gibbs, mu_closed):.1e}, {relative_error(g_per_particle, mu_closed):.1e}")

מסלול האנטרופיה הוא הפרש מרכזי לפי $N$, ולכן השגיאה שלו אמורה לרדת כמו ריבוע הצעד. מדדו את
סדר ההתכנסות — זו המדידה הראשונה של המעבדה.

In [ ]:
stencil = convergence_study(
    lambda k: chemical.mu_from_entropy(argon, energy, volume, N_GAS, dn=N_GAS / k),
    [4, 8, 16, 32, 64], mu_closed)
ratios = stencil.errors[:-1] / stencil.errors[1:]
print("dn = N/k for k =", stencil.refinements.astype(int).tolist())
print("relative errors:", ", ".join(f"{e:.2e}" for e in stencil.errors))
print(f"each halving divides the error by {ratios.min():.3f} to {ratios.max():.3f}")
print(f"observed order: {stencil.observed_order:.4f}")

שלוש התוצאות מסכימות, והערך של $\mu$ — $-16\,\kB T$ בקירוב — שלילי, משום שארגון בלחץ
$1\ \mathrm{bar}$ מדולל פי עשרה מיליון מהריכוז הקוונטי שלו.

## חלק 3 — שתי קופסאות מחליפות חלקיקים

שתי קופסאות של $10^5$ אתרים כל אחת, $2000$ חלקיקים, הרצפה של קופסה B מוגבהת בשיעור $2\,\kB T$,
וכל החלקיקים מתחילים בקופסה B. זמן הרלקסציה בגבול המדולל, $\tau$ — הנמדד בקפיצות מוצעות — קובע
כמה זמן להריץ.

### לנבא

היכן יתייצב מספר החלקיקים בקופסה A? שרטטו את $\mu_A - \mu_B$ כפונקציה של הזמן.

**הניבוי שלכם:**

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
SITES, N_TOTAL, OFFSET = 100_000, 2000, 2.0
boxes = chemical.two_boxes(SITES, SITES, 0.0, OFFSET * KT)
tau = chemical.relaxation_steps(boxes, N_TOTAL, T)
n_steps = int(20 * tau)
print(f"relaxation time tau = {tau:.0f} proposed hops; running {n_steps} per run")

trace = chemical.particle_exchange_sim(boxes, [0, N_TOTAL], T, n_steps,
                                       np.random.default_rng(0), record_every=20)
exact = chemical.exact_count_distribution(boxes, N_TOTAL, T)
dilute = chemical.equilibrium_split(boxes, N_TOTAL, T)
mus = trace.chemical_potentials / KT

fig, (top, bottom) = plt.subplots(2, 1, figsize=(8, 5.5), sharex=True)
top.plot(trace.steps / tau, trace.counts[:, 0], color="#2563eb", lw=1)
top.axhline(exact.mean, color="k", lw=1, label="exact equilibrium")
top.set_ylabel("N_A")
top.legend()
bottom.plot(trace.steps / tau, mus[:, 0] - mus[:, 1], color="#dc2626", lw=1)
bottom.axhline(0, color="k", lw=1)
bottom.set_ylabel("(mu_A - mu_B) / k_B T")
bottom.set_xlabel("t / tau")
plt.tight_layout()
plt.show()

settled = trace.tail(0.5)
print(f"second half: N_A = {settled[:, 0].mean():.1f}, N_B = {settled[:, 1].mean():.1f}")
print(f"exact: N_A = {exact.mean:.1f} +/- {exact.std:.1f};  dilute formula: {dilute[0]:.1f}")
densities = settled.mean(axis=0) / boxes.sites
print(f"density ratio n_A/n_B = {densities[0] / densities[1]:.2f}  (e^2 = {np.exp(2):.2f})")
half = len(trace.steps) // 2
late_gap = np.mean(mus[half:, 0] - mus[half:, 1])
print(f"mean of mu_A - mu_B over the second half: {late_gap:+.4f} k_B T")

ריצה אחת היא דגימה אחת. חזרו עליה עם שמונה מחוללי מספרים אקראיים בלתי תלויים, והשוו את הממוצע
להתפלגות המדויקת — תוצאה שחורגת ממנה ביותר מכשלוש שגיאות תקן תהיה באג, לא פיזיקה.

In [ ]:
def settled_count(rng):
    run = chemical.particle_exchange_sim(boxes, [0, N_TOTAL], T, n_steps, rng, record_every=20)
    return float(run.tail(0.5)[:, 0].mean())


exchange_study = seed_study(settled_count, n_seeds=8)
print(f"N_A over 8 seeds: {exchange_study.mean:.1f} +/- {exchange_study.standard_error:.1f}")
print(f"exact mean:       {exact.mean:.1f}")
print("agrees within 3 standard errors:", exchange_study.agrees_with(exact.mean))

## חלק 4 — שני ניסויי הפרכה

**נגד כלל הריכוז.** התחילו את אותן שתי קופסאות בצפיפויות *שוות*, $1000$ חלקיקים בכל אחת. אילו
חלקיקים היו זורמים במורד הפרש הריכוזים, שום דבר לא היה קורה.

**נגד "הפוטנציאל הכימי הוא האנרגיה לחלקיק".** הסירו את הרצפות. לקופסה A יש $10^4$ אתרים, לקופסה
B יש $10^5$, וכל אחת מתחילה עם $1000$ חלקיקים: לכל חלקיק אותה אנרגיה, אפס, בכל אחת מהקופסאות.

### לנבא

בכל אחד מהניסויים: האם משהו זורם, ולאיזה כיוון?

**הניבוי שלכם:**

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
equal_start = chemical.particle_exchange_sim(boxes, [1000, 1000], T, n_steps,
                                             np.random.default_rng(1), record_every=20)
start_mu = equal_start.chemical_potentials[0] / KT
end_counts = equal_start.tail(0.5).mean(axis=0)
print("equal densities, B's floor raised by 2 k_B T:")
print(f"  start: mu_A = {start_mu[0]:+.2f}, mu_B = {start_mu[1]:+.2f} k_B T")
print(f"  end:   N_A = {end_counts[0]:.0f}, N_B = {end_counts[1]:.0f}")

no_floors = chemical.two_boxes(10_000, 100_000, 0.0, 0.0)
tau_nf = chemical.relaxation_steps(no_floors, 2000, T)
same_energy = chemical.particle_exchange_sim(no_floors, [1000, 1000], T, int(20 * tau_nf),
                                             np.random.default_rng(2), record_every=20)
start_mu_nf = same_energy.chemical_potentials[0] / KT
end_nf = same_energy.tail(0.5).mean(axis=0)
exact_nf = chemical.exact_count_distribution(no_floors, 2000, T).mean
print("no floors, box A ten times smaller, equal numbers:")
print(f"  start: mu_A = {start_mu_nf[0]:+.2f}, mu_B = {start_mu_nf[1]:+.2f} k_B T")
print(f"  end:   N_A = {end_nf[0]:.0f} (exact {exact_nf:.1f}), N_B = {end_nf[1]:.0f}")
print(f"  densities: {end_nf[0] / 10_000:.4f} and {end_nf[1] / 100_000:.4f}")

# The falsifying experiments: particles flowed with no concentration difference at all, and
# with no energy difference at all -- each time from the higher mu to the lower.
assert start_mu[1] > start_mu[0] and end_counts[0] > 1500
assert start_mu_nf[0] > start_mu_nf[1] and end_nf[0] < 300

בניסוי הראשון חלקיקים זרמו מקופסה צפופה בדיוק כמו האחרת; בשני — בין קופסאות שבהן לכל חלקיק
הייתה אותה אנרגיה. בכל פעם הם עברו מהקופסה שבה $\mu$ גבוה יותר אל זו שבה הוא נמוך יותר.

## חלק 5 — הצורה הפונקציונלית

העבירו את הרצפה של B על פני התחום שבין $0$ לבין $3\,\kB T$, ושרטטו את $\ln(n_A/n_B)$ בשיווי משקל
כפונקציה של ההיסט.

### לנבא

איזו צורה, ואיזה שיפוע?

**הניבוי שלכם:**

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
offsets = np.arange(0.0, 3.01, 0.5)
log_ratios, log_errors = [], []
for offset in offsets:
    swept = chemical.two_boxes(SITES, SITES, 0.0, offset * KT)
    steps = int(20 * chemical.relaxation_steps(swept, N_TOTAL, T))

    def log_ratio(rng, swept=swept, steps=steps):
        n_a = chemical.particle_exchange_sim(swept, [0, N_TOTAL], T, steps, rng,
                                             record_every=20).tail(0.5)[:, 0].mean()
        return float(np.log(n_a / (N_TOTAL - n_a)))

    study = seed_study(log_ratio, n_seeds=4, base_seed=100)
    log_ratios.append(study.mean)
    log_errors.append(study.standard_error)

(sweep_slope, sweep_intercept), sweep_cov = np.polyfit(offsets, log_ratios, 1, cov=True)
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.errorbar(offsets, log_ratios, yerr=log_errors, fmt="o", color="#2563eb")
ax.plot(offsets, offsets, "k--", label="slope 1: n_A/n_B = exp(Delta u / k_B T)")
ax.set_xlabel("(u_B - u_A) / k_B T")
ax.set_ylabel("ln(n_A / n_B)")
ax.legend()
plt.tight_layout()
plt.show()
print(f"fitted slope: {sweep_slope:.4f} +/- {np.sqrt(sweep_cov[0, 0]):.4f} per k_B T")

כל $\kB T$ נוסף של היסט מכפיל את יחס הצפיפויות בגורם $e$: $n_A/n_B = e^{(u_B - u_A)/\kB T}$,
כלומר $\mu$ שווה בשתי הקופסאות, בגבול המדולל.

## חלק 6 — עד כמה יציב השוויון של הפוטנציאלים הכימיים?

בשיווי משקל ההפרש $\mu_A - \mu_B$ מרצד סביב אפס. ההתפלגות המדויקת של הפיצול נותנת את סטיית התקן
שלו עבור כל $N$, בלי סימולציה; הניבוי של בעיה 4 בדף הבעיות הוא $\kB T/\sqrt{Np(1-p)}$, כאשר $p$
הוא החלק מהחלקיקים שנמצא בקופסה A.

In [ ]:
sizes = [100, 1_000, 10_000, 100_000]
p_a = 1 / (1 + np.exp(-OFFSET))
spreads = []
for n in sizes:
    sized = chemical.two_boxes(100 * n, 100 * n, 0.0, OFFSET * KT)
    spreads.append(chemical.exact_count_distribution(sized, n, T).mu_difference_std() / KT)
    print(f"N = {n:>7,d}   std(mu_A - mu_B) = {spreads[-1]:.5f} k_B T   "
          f"(1/sqrt(N p (1-p)) = {1 / np.sqrt(n * p_a * (1 - p_a)):.5f})")
fluctuation_exponent = scaling_exponent(sizes, spreads)
print(f"fitted exponent: {fluctuation_exponent:.4f}  (the argument says -1/2)")

## חלק 7 — העמוד הברומטרי

ערמו שתים-עשרה קופסאות של $50\,000$ אתרים לעמוד, כשהרצפה של כל שכבה גבוהה בשיעור $0.4\,\kB T$
מזו של השכבה שמתחתיה — כשם שאנרגיית הכובד של שכבת אוויר גבוהה בשיעור $m g\,\Delta z$ מזו של
השכבה שמתחתיה. הכניסו את כל $2000$ החלקיקים לשכבה התחתונה ותנו להם לקפוץ.

### לנבא

לאיזה פרופיל הם יתייצבו? מה שווה בכל השכבות?

**הניבוי שלכם:**

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
LAYERS, LAYER_STEP = 12, 0.4
tower = chemical.column(LAYERS, 50_000, LAYER_STEP * KT)
start = [N_TOTAL] + [0] * (LAYERS - 1)


def column_run(rng):
    return chemical.particle_exchange_sim(tower, start, T, 400_000, rng, record_every=500)


column_runs = [column_run(np.random.default_rng(s)) for s in range(4)]
profile = np.mean([run.tail(0.5).mean(axis=0) for run in column_runs], axis=0)
layer_mu = np.mean([run.chemical_potentials[len(run.steps) // 2:].mean(axis=0)
                    for run in column_runs], axis=0) / KT
(column_slope, _), column_cov = np.polyfit(np.arange(10), np.log(profile[:10]), 1, cov=True)

fig, (left, right) = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
layers = np.arange(LAYERS)
left.barh(layers, profile, color="#2563eb", alpha=0.8, label="simulated")
left.plot(chemical.equilibrium_split(tower, N_TOTAL, T), layers, "k--",
          label="n(0) exp(-m g z / k_B T)")
left.set_xlabel("particles in layer")
left.set_ylabel("layer (height in units of k_B T / (0.4 m g))")
left.legend()
right.plot(layer_mu, layers, "o", color="#dc2626")
right.set_xlabel("mu / k_B T")
plt.tight_layout()
plt.show()

print(f"fitted decay per layer: {column_slope:.4f} +/- {np.sqrt(column_cov[0, 0]):.4f}"
      f"   (barometric formula: {-LAYER_STEP})")
print(f"spread of the layers' mu: {np.ptp(layer_mu):.3f} k_B T")

אותו פרופיל נובע מתוך $\mu$ לבדו. עבור אוויר אמיתי — חנקן בטמפרטורה $288\ \mathrm{K}$ — הפונקציה
`barometric_profile` מקבעת את $\mu$ בגובה הקרקע, ושואלת בכל גובה מהי הצפיפות שנותנת את אותו $\mu$
לאחר שמוסיפים את $m g z$. השוו אותו לגורם בולצמן של מודול 11, $e^{-m g z/\kB T}$.

In [ ]:
NITROGEN = 28.0134 * AMU
heights = np.linspace(0.0, 30_000.0, 7)
from_mu = chemical.barometric_profile(heights, 288.0, NITROGEN, 2.5e25)
from_boltzmann = 2.5e25 * np.exp(-NITROGEN * chemical.STANDARD_GRAVITY * heights / (K_B * 288.0))
for z, a, b in zip(heights, from_mu, from_boltzmann, strict=True):
    print(f"z = {z / 1000:5.1f} km   constant mu: {a:.4e}   Boltzmann: {b:.4e}   "
          f"ratio {a / b:.15f}")
barometric_gap = float(np.max(np.abs(from_mu / from_boltzmann - 1)))

שתי גזירות, אטמוספרה אחת.

## חלק 8 — אתר הספיחה

משטח של $200$ אתרי ספיחה באנרגיה $\varepsilon = -3\,\kB T$ מחליף חלקיקים עם מאגר של $20\,000$
אתרים באנרגיה $0$. הוספת חלקיקים מעלה את $\mu$ של המאגר. המשקל הגרנד-קנוני מנבא שכל אתר במשטח
מאוכלס בהסתברות
$1/(e^{(\varepsilon - \mu)/\kB T} + 1)$ — יהיה גודל המאגר אשר יהיה.

### לנבא

באיזה ערך של $\mu$ של המאגר מכוסה מחצית מהמשטח?

**הניבוי שלכם:**

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
SURFACE, RESERVOIR, EPSILON = 200, 20_000, -3.0 * KT
totals = [20, 60, 150, 300, 600, 1000, 1600, 2400]
adsorption = []
for total in totals:
    pair = chemical.two_boxes(SURFACE, RESERVOIR, EPSILON, 0.0)
    mu_common, _ = chemical.occupation_equilibrium(pair, total, T)
    exact_cover = chemical.exact_count_distribution(pair, total, T).mean / SURFACE
    run = chemical.particle_exchange_sim(pair, [0, total], T, 300_000,
                                         np.random.default_rng(total), record_every=50)
    simulated = float(run.tail(0.5)[:, 0].mean() / SURFACE)
    formula = chemical.site_occupation(mu_common, T, EPSILON)
    adsorption.append((mu_common, exact_cover, formula, simulated))
    print(f"N = {total:5d}   mu = {mu_common / KT:+.3f} k_B T   exact {exact_cover:.4f}   "
          f"formula {formula:.4f}   simulated {simulated:.4f}")

mu_values = np.array([row[0] for row in adsorption]) / KT
grid = np.linspace(-8, 0, 300)
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(grid, chemical.site_occupation(grid * KT, T, EPSILON), "k", label="1/(exp((eps-mu)/kT)+1)")
ax.plot(mu_values, [row[3] for row in adsorption], "o", color="#2563eb", label="simulated")
ax.axvline(EPSILON / KT, color="grey", ls=":")
ax.set_xlabel("mu / k_B T")
ax.set_ylabel("fraction of surface covered")
ax.legend()
plt.tight_layout()
plt.show()
occupation_gap = max(abs(row[1] - row[2]) for row in adsorption)

המשטח מגיב לערך של $\mu$ ולשום דבר אחר. כשכותבים אותה במונחי לחץ הגז, אותה עקומה היא
איזותרמת לנגמיור $\theta = P/(P + P_0)$.

## חלק 9 — אוסמוזה, ומדידה

קרום חדיר-למחצה מעביר מים אך לא סוכר. במודל התמיסה האידיאלית, שוויון הפוטנציאל הכימי של המים
משני צדיו מחייב את $\Pi v = -\kB T\ln(1 - x)$, שהגבול המדולל שלו הוא הנוסחה של ואן 'ט הוף,
$\Pi = n_s\kB T$. השוו תחילה בין השתיים; אחר כך התאימו קריאות סינתטיות.

> **על הקריאות.** הקריאות שלהלן **סינתטיות**: הן נוצרו על ידי
> `chemical.osmosis_readings` מתוך המודל הזה עם שגיאה אקראית של $3\%$, והן באות במקום מדידות
> של כיתה בצינוריות דיאליזה. הן יכולות רק לאשר את המודל שממנו באו. כדי להשתמש בנתונים שלכם,
> החליפו את שני המערכים.

In [ ]:
T_ROOM = 298.15
v_water = chemical.WATER_MOLECULAR_VOLUME
print("c [mol/L]   van 't Hoff [MPa]   -(kT/v) ln(1-x) [MPa]   difference")
for molar in (0.01, 0.1, 0.5, 1.0):
    solute = molar * 1e3 * N_A
    dilute_pi = chemical.osmotic_pressure(solute, T_ROOM)
    lattice_pi = chemical.lattice_osmotic_pressure(solute * v_water, v_water, T_ROOM)
    print(f"{molar:9.2f}   {dilute_pi / 1e6:17.4f}   {lattice_pi / 1e6:21.4f}   "
          f"{lattice_pi / dilute_pi - 1:.3%}")

concentrations = np.array([0.05, 0.1, 0.2, 0.3, 0.4, 0.5])  # mol/L
readings = chemical.osmosis_readings(concentrations, T_ROOM, 0.03, np.random.default_rng(13))

(rt_slope, rt_intercept), rt_cov = np.polyfit(concentrations * 1e3, readings, 1, cov=True)
rt_error = float(np.sqrt(rt_cov[0, 0]))
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(concentrations, readings / 1e6, "o", color="#2563eb", label="synthetic readings")
ax.plot(concentrations, (rt_slope * concentrations * 1e3 + rt_intercept) / 1e6, "k--", label="fit")
ax.set_xlabel("concentration [mol/L]")
ax.set_ylabel("osmotic pressure [MPa]")
ax.legend()
plt.tight_layout()
plt.show()
print(f"fitted slope R T = {rt_slope:.0f} +/- {rt_error:.0f} J/mol;  R T at {T_ROOM} K = "
      f"{R_GAS * T_ROOM:.0f} J/mol")
print(f"R estimated from sugar water: {rt_slope / T_ROOM:.2f} +/- {rt_error / T_ROOM:.2f} "
      "J/(mol K)")

## חלק 10 — בדיקות אוטומטיות

חישוב שלא בדקתם הוא תמונה, לא ראיה. אלה אותן טענות בדיקה שרצות בחבילת הבדיקות של הפרויקט.

In [ ]:
# 1. Three routes to mu agree (Part 2), and the entropy stencil is second order.
assert relative_error(mu_entropy, mu_closed) < 1e-6
assert relative_error(mu_gibbs, mu_closed) < 1e-6
assert relative_error(g_per_particle, mu_closed) < 1e-9
assert abs(stencil.observed_order - 2.0) < 0.05

# 2. The simulated split agrees with the exact distribution (Part 3).
assert exchange_study.agrees_with(exact.mean)

# 3. Each extra k_B T of offset multiplies the density ratio by e (Part 5).
assert abs(sweep_slope - 1.0) < 5 * np.sqrt(sweep_cov[0, 0]) + 0.02

# 4. Fluctuations of mu_A - mu_B fall as N^(-1/2) (Part 6).
assert abs(fluctuation_exponent + 0.5) < 0.01

# 5. The column is barometric, with one mu at every height (Part 7).
assert abs(column_slope + LAYER_STEP) < 0.01
assert np.ptp(layer_mu) < 0.1
assert barometric_gap < 1e-12

# 6. The one-site formula reproduces the exact coverage (Part 8).
assert occupation_gap < 1e-3

# 7. The fitted R T is within four standard errors of the true value (Part 9).
assert abs(rt_slope - R_GAS * T_ROOM) < 4 * rt_error

print("all checks passed")

## חלק 11 — חקרו בעצמכם

קבעו את ההיסט בין הרצפות, את מספר החלקיקים ואת הגדלים של שתי הקופסאות, בחרו כיצד החלקיקים
מתחילים, ואז לחצו **Run Interact**. הלוח השמאלי מציג את מספר החלקיקים בכל קופסה כפונקציה של
הזמן; הלוח הימני מציג את שני הפוטנציאלים הכימיים. שלושה דברים שכדאי לנסות:

1. עשו את B גדולה פי עשרה מ-A בלי היסט, והתחילו עם כל החלקיקים ב-B. היכן זה מתייצב, ומדוע?
2. בחרו היסט ויחס גדלים שמבטלים זה את זה בדיוק, כך שבגבול המדולל הפיצול יהיה שווה. איזה צירוף
   עושה זאת?
3. מלאו את הקופסאות עד כמה אחוזים. כמה רחוק סוטה מספר החלקיקים המתייצב מהנוסחה של הגבול
   המדולל, והאם הוא סוטה מהנוסחה המדויקת?

In [ ]:
import ipywidgets as widgets


def explore(offset=2.0, log10_particles=3.0, sites_a=100_000, sites_b=100_000, start="all in B"):
    n = int(round(10**log10_particles))
    pair = chemical.two_boxes(int(sites_a), int(sites_b), 0.0, offset * KT)
    if n >= pair.total_sites:
        print("more particles than sites")
        return
    initial = {"all in B": [0, n], "all in A": [n, 0], "equal numbers": [n // 2, n - n // 2]}[start]
    if initial[0] > sites_a or initial[1] > sites_b:
        print("the starting box is too small for that many particles")
        return
    steps = int(15 * chemical.relaxation_steps(pair, n, T))
    run = chemical.particle_exchange_sim(pair, initial, T, steps, np.random.default_rng(7),
                                         record_every=max(1, steps // 400))
    mus = run.chemical_potentials / KT
    fig, (left, right) = plt.subplots(1, 2, figsize=(12, 3.8))
    left.plot(run.steps, run.counts[:, 0], label="N_A")
    left.plot(run.steps, run.counts[:, 1], label="N_B")
    left.axhline(chemical.equilibrium_split(pair, n, T)[0], color="k", ls="--", lw=1)
    left.set_xlabel("proposed hops")
    left.legend()
    right.plot(run.steps, mus[:, 0], label="mu_A")
    right.plot(run.steps, mus[:, 1], label="mu_B")
    right.set_xlabel("proposed hops")
    right.set_ylabel("mu / k_B T")
    right.legend()
    plt.tight_layout()
    plt.show()
    final = run.tail(0.5).mean(axis=0)
    print(f"settled: N_A = {final[0]:.1f}, N_B = {final[1]:.1f};  dilute formula "
          f"{chemical.equilibrium_split(pair, n, T)[0]:.1f};  exact "
          f"{chemical.exact_count_distribution(pair, n, T).mean:.1f}")


widgets.interact_manual(
    explore,
    offset=widgets.FloatSlider(value=2.0, min=0.0, max=10.0, step=0.25, description="u_B - u_A"),
    log10_particles=widgets.FloatSlider(value=3.0, min=2.0, max=4.3, step=0.1,
                                        description="log10 N"),
    sites_a=widgets.IntSlider(value=100_000, min=1_000, max=200_000, step=1_000,
                              description="sites A"),
    sites_b=widgets.IntSlider(value=100_000, min=1_000, max=200_000, step=1_000,
                              description="sites B"),
    start=widgets.Dropdown(options=["all in B", "all in A", "equal numbers"], description="start"),
);

## בחנו את הבנתכם

הריצו את התא שלהלן לחידון הנבדק אוטומטית. אותן שאלות, עם הסברים כתובים לכל אפשרות, נמצאות
בדף המודול.

In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "13-chemical-potential.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## לפני שאתם עוזבים

רשמו כמה משפטים על כל אחת, בתא שלהלן.

1. מה ניבאתם שהתברר כשגוי, ומה בדיוק היה הפגם בנימוקכם?
2. בחלק 4 חלקיקים זרמו בין שתי קופסאות בצפיפויות שוות, ובין שתי קופסאות שבהן לכל חלקיק הייתה
   אותה אנרגיה. מה, בכל אחד מהמקרים, עשה את $\mu$ של קופסה אחת לגבוה יותר?
3. העמוד של חלק 7 וגורם בולצמן של מודול 11 נותנים את אותה אטמוספרה. מה מניחה כל אחת מהגזירות?
4. מדוע למשטח הסופח לא היה אכפת כמה גדול המאגר?

**התשובות שלכם:**

1.
2.
3.
4.